### Change directory:

In [1]:
import os

In [2]:
%pwd

'c:\\Users\\paric\\Documents\\NLP\\Text-Summarizer\\research'

In [3]:
# Change directory:
os.chdir("../")

In [4]:
%pwd

'c:\\Users\\paric\\Documents\\NLP\\Text-Summarizer'

## 1. Update the config.yaml 

Take dataset from ingestion output, transform it using Pegasus tokenizer, and save results in a separate folder.

In [5]:
# data_transformation: 
#    root_dir: artifacts/data_transformation # All transformed data (output) will be saved here
#    data_path: artifacts/data_ingestion/samsum_dataset # Data Transformation will take data from here (input data path)
#    tokenizer_name: google/pegasus-cnn_dailymail # Use this Hugging Face model tokenizer name
#    tokenizer_name: T5-small # Use this Hugging Face model tokenizer name

## 2. Update the params.py

In this case, we dont have any parameters. Whenever we define the model configuration, we usually include parameters, but at this stage, they are not required.

## 3. Update/Create the Entity:

In [6]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class DataTransformationConfig:
    root_dir: Path
    data_path: Path
    tokenizer_name: Path

## 4. Update the Configuration Manager:

In [7]:
from textsummarizer.constants import *
from textsummarizer.utils.common import read_yaml, create_directories

In [8]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])


    # This defines a method inside a class, build configuration for data transformation stage
    def get_data_transformation_config(self) -> DataTransformationConfig:
        config = self.config.data_transformation # Reads the data_transformation section from YAML config and config holds all transformation settings.

        create_directories([config.root_dir]) # Ensures the output folder exists. If not, it creates it

        data_transformation_config = DataTransformationConfig( # Builds a structured configuration object. Transfers values from YAML → Python object
            root_dir=config.root_dir, # where transformed data will be saved
            data_path=config.data_path, # input data from ingestion stage
            tokenizer_name = config.tokenizer_name # Hugging Face tokenizer to use
        )

        return data_transformation_config # Sends the configuration to the pipeline, So the next stage (Data Transformation component) can use it


## 5. Update the Components:

In [9]:
import os
from textsummarizer.logging import logger
from transformers import AutoTokenizer # loads pretrained tokenizer from Hugging Face
from datasets import load_dataset, load_from_disk # loads dataset (from internet or local disk)

c:\Users\paric\Documents\NLP\Text-Summarizer\texts\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [10]:
# # This class handles data preprocessing and tokenization
# class DataTransformation:
#     def __init__(self, config: DataTransformationConfig): # This is the constructor. It runs automatically when the class is created. It receives configuration settings (config)
#         self.config = config # Saves the config inside the class. So we can use paths and settings later
#         self.tokenizer = AutoTokenizer.from_pretrained(config.tokenizer_name) # Loads a pretrained tokenizer from Hugging Face. Converts text into numbers (tokens) for the model

#     def convert_examples_to_features(self,example_batch): # This function converts raw text into model-ready format. It processes a batch of data (multiple examples at once)
#         input_encodings = self.tokenizer(example_batch['dialogue'] , max_length = 1024, truncation = True ) # Takes the dialogue (input text). Converts it into numbers (tokens). max_length=1024 → limits long text. truncation=True → cuts extra text if too long. Output = model input format
        
#         with self.tokenizer.as_target_tokenizer(): # Switch to target mode (summary): Tells tokenizer: “now we are processing output text (summary)”. Important for sequence-to-sequence models like Pegasus
#             target_encodings = self.tokenizer(example_batch['summary'], max_length = 128, truncation = True ) # Encode input text (dialogue):Takes the dialogue (input text), Converts it into numbers (tokens), max_length=1024 → limits long text, truncation=True → cuts extra text if too long. Output = model input format

#         # The dialogue converted into numbers. This is what the model reads. attention_mask, 1 = real words, 0 = padding (ignore this). Helps the model focus only on real text. The summary converted into numbers.  This is the correct answer the model should learn to produce. input_ids → question (dialogue), labels → answer (summary), attention_mask → tells what to ignore    
#         return { # Return final format 
#             'input_ids' : input_encodings['input_ids'], # encoded dialogue (input to model)
#             'attention_mask': input_encodings['attention_mask'], # tells model which words matter (1 = real, 0 = padding)
#             'labels': target_encodings['input_ids'] # correct summary (what model should learn to produce)
#         }
    

#     def convert(self):
#         dataset_samsum = load_from_disk(self.config.data_path) # Loads the dataset from your local folder. This is the output of data ingestion stage
#         dataset_samsum_pt = dataset_samsum.map(self.convert_examples_to_features, batched = True) # Apply transformation, Applies your tokenizer function to every example. Converts:dialogue → input_ids, summary → labels, batched=True: means it processes multiple rows at once (faster)
#         dataset_samsum_pt.save_to_disk(os.path.join(self.config.root_dir,"samsum_dataset")) # Save transformed data: Saves the processed dataset. This data is now ready for model training

In [11]:
# class DataTransformation:

#     def __init__(self, config):
#         self.config = config

#         self.tokenizer = AutoTokenizer.from_pretrained(
#             config.tokenizer_name
#         )

#     def convert_examples_to_features(self, example_batch):

#         # Tokenize dialogue
#         input_encodings = self.tokenizer(
#             example_batch['dialogue'], # Converts input conversation into tokens.
#             max_length=1024, 
#             truncation=True,
#             padding='max_length'
#         )

#         # Tokenize summary
#         target_encodings = self.tokenizer(
#             text_target=example_batch['summary'], # Converts target summary into labels.
#             max_length=128,
#             truncation=True,
#             padding='max_length' # Makes all sequences same length.Needed for batching during training.
#         )

#         return {
#             'input_ids': input_encodings['input_ids'],
#             'attention_mask': input_encodings['attention_mask'],
#             'labels': target_encodings['input_ids']
#         }

#     def convert(self):

#         print("Loading dataset...")

#         dataset_samsum = load_from_disk(
#             self.config.data_path
#         )

#         print("Transforming dataset...")

#         dataset_samsum_pt = dataset_samsum.map(
#             self.convert_examples_to_features,
#             batched=True
#         )

#         print("Saving transformed dataset...")

#         dataset_samsum_pt.save_to_disk(
#             self.config.root_dir
#         )

#         print("Transformation completed successfully.")


In [12]:
class DataTransformation:
    # This class handles converting raw dataset into tokenized format for training

    def __init__(self, config):
        # Store configuration (paths, tokenizer name, etc.)
        self.config = config

        # Load tokenizer (must match the model, e.g., t5-small)
        self.tokenizer = AutoTokenizer.from_pretrained(config.tokenizer_name)

    def convert_examples_to_features(self, example_batch):

        # ------------------------------------------------------------
        # STEP 1: Prepare input text (IMPORTANT FOR T5 MODEL)
        # T5 needs a task prefix like "summarize:"
        # ------------------------------------------------------------
        input_texts = [
            "summarize: " + text for text in example_batch['dialogue']
        ]

        # ------------------------------------------------------------
        # STEP 2: Tokenize input text (dialogue)
        # Converts text → token IDs + attention mask
        # max_length=1024 limits input size
        # truncation=True cuts long texts
        # padding='max_length' makes all sequences equal length
        # ------------------------------------------------------------
        input_encodings = self.tokenizer(
            input_texts,
            max_length=1024,
            truncation=True,
            padding='max_length'
        )

        # ------------------------------------------------------------
        # STEP 3: Tokenize target text (summary)
        # text_target is used for encoder-decoder models like T5
        # ------------------------------------------------------------
        target_encodings = self.tokenizer(
            text_target=example_batch['summary'],
            max_length=128,
            truncation=True,
            padding='max_length'
        )

        # ------------------------------------------------------------
        # STEP 4: Extract labels (target token IDs)
        # These are what the model will learn to predict
        # ------------------------------------------------------------
        labels = target_encodings["input_ids"]

        # ------------------------------------------------------------
        # STEP 5: Return final processed features
        # input_ids → model input
        # attention_mask → tells model which tokens to attend to
        # labels → correct output for training
        # ------------------------------------------------------------
        return {
            'input_ids': input_encodings['input_ids'],
            'attention_mask': input_encodings['attention_mask'],
            'labels': labels
        }

    def convert(self):

        # ------------------------------------------------------------
        # STEP 6: Load raw dataset from disk
        # This should be ORIGINAL dataset (before tokenization)
        # ------------------------------------------------------------
        print("Loading dataset...")
        dataset_samsum = load_from_disk(self.config.data_path)

        # ------------------------------------------------------------
        # STEP 7: Apply tokenization to entire dataset
        # batched=True improves speed significantly
        # ------------------------------------------------------------
        print("Transforming dataset...")
        dataset_samsum_pt = dataset_samsum.map(
            self.convert_examples_to_features,
            batched=True
        )

        # ------------------------------------------------------------
        # STEP 8: Save processed dataset
        # This dataset will be used for training
        # ------------------------------------------------------------
        print("Saving transformed dataset...")
        dataset_samsum_pt.save_to_disk(self.config.root_dir)

        print("Transformation completed successfully.")

## 6. Update the Pipelines:

In [13]:
try:
    config = ConfigurationManager()
    data_transformation_config = config.get_data_transformation_config()
    data_transformation = DataTransformation(config=data_transformation_config)
    data_transformation.convert()
except Exception as e:
    raise e

[2026-05-13 15:01:51,299: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-05-13 15:01:51,303: INFO: common: yaml file: params.yaml loaded successfully]
[2026-05-13 15:01:51,304: INFO: common: created directory at: artifacts]
[2026-05-13 15:01:51,310: INFO: common: created directory at: artifacts/data_transformation]
[2026-05-13 15:01:51,551: INFO: _client: HTTP Request: HEAD https://huggingface.co/t5-small/resolve/main/config.json "HTTP/1.1 200 OK"]
[2026-05-13 15:01:51,664: INFO: _client: HTTP Request: HEAD https://huggingface.co/t5-small/resolve/main/tokenizer_config.json "HTTP/1.1 200 OK"]
[2026-05-13 15:01:51,774: INFO: _client: HTTP Request: GET https://huggingface.co/api/models/t5-small/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 307 Temporary Redirect"]
[2026-05-13 15:01:51,863: INFO: _client: HTTP Request: GET https://huggingface.co/api/models/google-t5/t5-small/tree/main/additional_chat_templates?recursive=false&expand=fals

Map: 100%|██████████| 818/818 [00:00<00:00, 2359.70 examples/s]


Saving transformed dataset...


Saving the dataset (1/1 shards): 100%|██████████| 818/818 [00:00<00:00, 71416.93 examples/s]

Transformation completed successfully.
